# Nuclear / Droplet Segmentation Pipeline — v1

**Architecture overview**

- U-Net inference with overlapping patch stitching
- Per-object probability normalisation (post-inference, pre-threshold) so that binary mask generation is uniform regardless of internal signal variation
- Focus-aware Z selection (object-based nuclear scorer, droplet-resistant)
- Hybrid DBSCAN + Hungarian tracking across time
- Halo / ring / N:C ratio measurement on **original unmodified images**
- All intermediate artefacts written to disk; stage flags control re-computation

**Run order:** top-to-bottom on first run.  
After that, flip stage flags in **Section 22** and rerun only what you need.

**Path convention:** every path is resolved through `cfg` properties — nothing is hardcoded in execution cells.

## Section 1 — Imports

In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import time
from dataclasses import dataclass, asdict, field
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union
import shutil

from joblib import Parallel, delayed
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
from skimage import filters, measure, morphology

try:
    import tensorflow as tf
except Exception:
    tf = None
    print("WARNING: TensorFlow not available — segmentation will not run.")

try:
    import tifffile as tiff
except Exception:
    tiff = None
    print("WARNING: tifffile not available — TIFF I/O disabled.")

try:
    from sklearn.cluster import DBSCAN
except Exception:
    DBSCAN = None
    print("WARNING: scikit-learn not available — DBSCAN tracking disabled.")

print("Imports complete.")

## Section 2 — Configuration

In [ ]:
def get_n_workers() -> int:
    return int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 1))


@dataclass
class PipelineConfig:
    # ── Root paths ────────────────────────────────────────────────────────
    project_root:     str = "/home/tdeibert/Data/Machine_Learning_Dev/"
    model_subdir:     str = "Models"
    model_name:       str = "unet_droplet_nucleus_v6_2_final.keras"
    image_subdir:     str = "Images"
    input_image_name: str = "control_extract_1.1.tif"
    output_subdir:    str = "Outputs"

    # ── Imaging ───────────────────────────────────────────────────────────
    pixel_size_um:           float = 0.108
    z_step_um:               float = 1.0
    nuclear_channel_index:   int   = 1
    membrane_channel_index:  int   = 0

    # ── Halo analysis ─────────────────────────────────────────────────────
    halo_step_px:  int = 4
    n_halos:       int = 4
    erosion_px:    int = 4

    # ── Grouping / tracking ───────────────────────────────────────────────
    z_group_tolerance_um:      float = 1.0
    time_track_tolerance_um:   float = 5.0
    multi_nucleus_exclusion_um: float = 1.0

    track_dbscan_eps_um:         float = 6.0
    track_dbscan_min_samples:    int   = 1
    track_crowded_distance_scale: float = 0.65
    track_area_log_weight:       float = 12.0
    track_area_ratio_max:        float = 5.0

    # ── Stitched acquisition layout ───────────────────────────────────────
    tile_rows:         int   = 2
    tile_cols:         int   = 3
    minutes_per_tile:  float = 1.0
    serpentine_scan:   bool  = True

    # ── Segmentation ──────────────────────────────────────────────────────
    min_nucleus_area_px:       int   = 50
    nucleus_hole_size_px:      int   = 200
    nucleus_opening_radius_px: int   = 1
    min_nucleus_circularity:   float = 0.45
    batch_size:                int   = 1
    use_mixed_precision:       bool  = False
    patch_size:                int   = 512
    patch_stride:              int   = 256
    background_class_index:    int   = 0
    droplet_class_index:       int   = 1
    nucleus_class_index:       int   = 2

    # ── Per-object probability normalisation ──────────────────────────────
    # Coarse threshold used only to find object extents for normalisation.
    # Keep intentionally loose — it does NOT set the final mask boundary.
    prob_norm_coarse_threshold:   float = 0.25
    # Final threshold applied to the normalised probability map.
    prob_norm_final_threshold:    float = 0.50
    # Which class channels to normalise. Background is left as-is.
    prob_norm_nucleus:            bool  = True
    prob_norm_droplet:            bool  = True
    # Minimum object area (px) accepted during coarse detection for normalisation.
    prob_norm_min_object_area_px: int   = 30

    # ── Focus-aware Z selection ───────────────────────────────────────────
    use_focus_z_selection:     bool  = True
    focus_window_radius:       int   = 1
    focus_edge_z_exclusion:    int   = 2
    focus_channel_index: Optional[int] = None
    focus_metric:              str   = "object_based_nuclear"
    focus_min_nucleus_area_um2: float = 20.0
    focus_max_nucleus_area_um2: float = 500.0
    focus_threshold_k:         float = 1.5
    focus_sharpness_weight:    float = 0.5

    # ── HPC ───────────────────────────────────────────────────────────────
    n_workers: Optional[int] = None

    def __post_init__(self):
        if self.n_workers is None:
            self.n_workers = min(get_n_workers(), 16)

    # ── Unit conversions (derived) ────────────────────────────────────────
    @property
    def z_group_tolerance_px(self) -> float:
        return self.z_group_tolerance_um / self.pixel_size_um

    @property
    def time_track_tolerance_px(self) -> float:
        return self.time_track_tolerance_um / self.pixel_size_um

    @property
    def multi_nucleus_exclusion_px(self) -> float:
        return self.multi_nucleus_exclusion_um / self.pixel_size_um

    @property
    def track_dbscan_eps_px(self) -> float:
        return self.track_dbscan_eps_um / self.pixel_size_um

    @property
    def effective_focus_channel_index(self) -> int:
        return self.nuclear_channel_index if self.focus_channel_index is None else self.focus_channel_index

    # ── Base paths ────────────────────────────────────────────────────────
    @property
    def project_root_path(self) -> Path:
        return Path(self.project_root)

    @property
    def model_path(self) -> Path:
        return self.project_root_path / self.model_subdir / self.model_name

    @property
    def input_image_path(self) -> Path:
        return self.project_root_path / self.image_subdir / self.input_image_name

    @property
    def output_dir(self) -> Path:
        return self.project_root_path / self.output_subdir

    # ── Output subdirectories ─────────────────────────────────────────────
    @property
    def seg_dir(self) -> Path:
        return self.output_dir / "segmentation"

    @property
    def obj_dir(self) -> Path:
        return self.output_dir / "objects"

    @property
    def track_dir(self) -> Path:
        return self.output_dir / "tracking"

    @property
    def analysis_dir(self) -> Path:
        return self.output_dir / "analysis"

    @property
    def qc_dir(self) -> Path:
        return self.output_dir / "qc"

    @property
    def mask_tif_dir(self) -> Path:
        return self.output_dir / "mask_tifs"

    # ── Segmentation artefact paths ───────────────────────────────────────
    @property
    def segmentation_index_path(self) -> Path:
        return self.seg_dir / "segmentation_index.pkl"

    @property
    def segmentation_class_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "segmentation_class_hyperstack.tif"

    @property
    def segmentation_label_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "segmentation_label_hyperstack.tif"

    @property
    def nucleus_instance_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "nucleus_instance_hyperstack.tif"

    @property
    def droplet_instance_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "droplet_instance_hyperstack.tif"

    @property
    def segmentation_probability_hyperstack_path(self) -> Path:
        """Raw U-Net probability output — preserved for provenance."""
        return self.mask_tif_dir / "segmentation_probability_hyperstack_raw.tif"

    @property
    def normalised_probability_hyperstack_path(self) -> Path:
        """Per-object normalised probability — used for mask generation."""
        return self.mask_tif_dir / "segmentation_probability_hyperstack_normalised.tif"

    @property
    def segmentation_roi_table_path(self) -> Path:
        return self.seg_dir / "segmentation_roi_table.pkl"

    def to_serializable_dict(self) -> Dict[str, object]:
        return asdict(self)

    def __repr__(self) -> str:
        keys = [
            "project_root", "model_name", "input_image_name", "output_subdir",
            "pixel_size_um", "nuclear_channel_index", "membrane_channel_index",
            "z_group_tolerance_um", "time_track_tolerance_um",
            "multi_nucleus_exclusion_um", "track_dbscan_eps_um",
            "min_nucleus_area_px", "patch_size", "patch_stride",
            "prob_norm_coarse_threshold", "prob_norm_final_threshold",
            "prob_norm_nucleus", "prob_norm_droplet",
            "n_workers", "use_mixed_precision",
        ]
        return "\n".join(f"{k}: {getattr(self, k)}" for k in keys)


cfg = PipelineConfig()

for p in [
    cfg.output_dir, cfg.seg_dir, cfg.obj_dir, cfg.track_dir,
    cfg.analysis_dir, cfg.qc_dir, cfg.mask_tif_dir,
]:
    p.mkdir(parents=True, exist_ok=True)

cfg

## Section 3 — Save / Load Configuration

In [ ]:
def save_config(config: PipelineConfig, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(config.to_serializable_dict(), indent=2))


def load_config(in_path: Path) -> PipelineConfig:
    data = json.loads(in_path.read_text())
    return PipelineConfig(**data)


save_config(cfg, cfg.output_dir / "pipeline_config.json")
print("Configuration saved.")

## Section 4 — GPU Setup

In [ ]:
def configure_tensorflow_for_gpu(config: PipelineConfig) -> None:
    if tf is None:
        print("TensorFlow not available — skipping GPU setup.")
        return

    gpus = tf.config.list_physical_devices("GPU")
    print("GPUs found:", gpus)

    if getattr(config, "use_mixed_precision", False):
        try:
            from tensorflow.keras import mixed_precision
            mixed_precision.set_global_policy("mixed_float16")
            print("Mixed precision enabled.")
        except Exception as exc:
            print("Could not enable mixed precision:", exc)

    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception as exc:
            print("Could not set memory growth:", exc)

## Section 5 — Image Loading Helpers

In [ ]:
def load_image_5d(image_path: Union[str, Path]) -> np.ndarray:
    """Load a TIFF and return a (T, Z, C, Y, X) array."""
    if tiff is None:
        raise ImportError("tifffile is required to load TIFF data.")
    arr = tiff.imread(image_path)
    print(f"Loaded image shape: {arr.shape}  dtype: {arr.dtype}")
    return arr


def get_nuclear_plane(img_5d: np.ndarray, t: int, z: int, config: PipelineConfig) -> np.ndarray:
    """Return the (Y, X) nuclear channel plane for a given T and Z."""
    return img_5d[t, z, config.nuclear_channel_index]


def get_membrane_plane(img_5d: np.ndarray, t: int, z: int, config: PipelineConfig) -> np.ndarray:
    """Return the (Y, X) membrane channel plane for a given T and Z."""
    return img_5d[t, z, config.membrane_channel_index]


def get_full_plane_yxc(img_5d: np.ndarray, t_idx: int, z_idx: int) -> np.ndarray:
    """Return a full multi-channel plane in (Y, X, C) format from a (T, Z, C, Y, X) image."""
    return np.moveaxis(img_5d[t_idx, z_idx], 0, -1)

## Section 6 — Focus-Scoring Utilities

In [ ]:
# ── Internal helpers ──────────────────────────────────────────────────────────

def _normalize_2d(img2d: np.ndarray) -> np.ndarray:
    """Normalise a 2-D plane to [0, 1] for focus scoring."""
    arr = np.asarray(img2d, dtype=np.float32)
    lo, hi = np.nanmin(arr), np.nanmax(arr)
    if not (np.isfinite(lo) and np.isfinite(hi) and hi > lo):
        return np.zeros_like(arr)
    return (arr - lo) / (hi - lo)


def _area_um2_to_px(area_um2: float, pixel_size_um: float) -> int:
    return max(1, int(np.ceil(area_um2 / pixel_size_um ** 2)))


# ── Scorer implementations ────────────────────────────────────────────────────

def focus_score_variance_laplacian(img2d: np.ndarray) -> float:
    """Whole-plane variance-of-Laplacian (fallback / debug)."""
    return float(np.nanvar(ndi.laplace(_normalize_2d(img2d))))


def focus_score_nucleus_weighted_laplacian(
    img2d: np.ndarray,
    candidate_percentile: float = 97.5,
    min_candidate_pixels: int = 50,
) -> float:
    """Bright-pixel-weighted Laplacian focus score."""
    img_f = _normalize_2d(img2d)
    finite = np.isfinite(img_f)
    if not finite.any():
        return 0.0
    cutoff = np.nanpercentile(img_f[finite], candidate_percentile)
    candidate_mask = (img_f >= cutoff) & finite
    if int(candidate_mask.sum()) < min_candidate_pixels:
        return 0.25 * focus_score_variance_laplacian(img2d)
    lap = ndi.laplace(img_f)
    weights = np.maximum(img_f[candidate_mask], 1e-6)
    weighted_energy = np.average(lap[candidate_mask] ** 2, weights=weights)
    signal_bonus = np.sqrt(max(float(candidate_mask.mean()), 1e-8))
    return float(weighted_energy * signal_bonus)


def focus_score_hybrid_nuclear_laplacian(
    img2d: np.ndarray,
    global_weight: float = 0.2,
    nuclear_weight: float = 0.8,
) -> float:
    """Hybrid score (retained for comparison / debugging)."""
    return float(
        global_weight * focus_score_variance_laplacian(img2d)
        + nuclear_weight * focus_score_nucleus_weighted_laplacian(img2d)
    )


def focus_score_object_based_nuclear(
    img2d: np.ndarray,
    pixel_size_um: float,
    min_nucleus_area_um2: float = 20.0,
    max_nucleus_area_um2: float = 500.0,
    blur_sigma: float = 1.5,
    threshold_k: float = 1.5,
    min_signal_fraction: float = 0.005,
    sharpness_weight: float = 0.5,
    min_object_count: int = 1,
) -> float:
    """
    Object-based nuclear focus score — droplet-resistant.

    Uses a two-band area filter (upper limit below droplet body area),
    mean+k*std thresholding (robust when nuclear signal is sparse), and
    a per-object sharpness weight to penalise blurry droplet halos.
    """
    nuc = _normalize_2d(img2d)
    if np.nanmax(nuc) <= 0:
        return 0.0

    mean_val = float(np.nanmean(nuc))
    if float((nuc > mean_val).mean()) < min_signal_fraction:
        return 0.0

    nuc_blur = filters.gaussian(nuc, sigma=blur_sigma, preserve_range=True)
    thr = mean_val + threshold_k * float(np.nanstd(nuc_blur))
    if thr >= 1.0:
        return 0.0

    min_area_px = _area_um2_to_px(min_nucleus_area_um2, pixel_size_um)
    max_area_px = _area_um2_to_px(max_nucleus_area_um2, pixel_size_um)

    mask = morphology.remove_small_objects(
        (nuc_blur > thr).astype(bool), min_size=min_area_px)

    lap = ndi.laplace(nuc)
    plane_lap_var = float(np.var(lap)) + 1e-12
    labeled = measure.label(mask)

    score = 0.0
    valid_count = 0

    for p in measure.regionprops(labeled, intensity_image=nuc):
        if p.area < min_area_px or p.area > max_area_px:
            continue
        roundness_weight = max(0.05, 1.0 - float(p.eccentricity))
        solidity_weight  = max(0.05, float(getattr(p, "solidity", 1.0)))
        base = float(p.area) * float(p.mean_intensity) * roundness_weight * solidity_weight
        if sharpness_weight > 0:
            obj_pixels = lap[labeled == p.label]
            sharpness = float(np.var(obj_pixels)) / plane_lap_var
            combined_weight = 1.0 + sharpness_weight * sharpness
        else:
            combined_weight = 1.0
        score += base * combined_weight
        valid_count += 1

    return float(score) if valid_count >= min_object_count else 0.0


# ── Dispatcher ────────────────────────────────────────────────────────────────

_FOCUS_METRICS = {
    "object_based_nuclear":       lambda img, pxum: focus_score_object_based_nuclear(img, pixel_size_um=pxum),
    "variance_laplacian":         lambda img, pxum: focus_score_variance_laplacian(img),
    "nucleus_weighted_laplacian": lambda img, pxum: focus_score_nucleus_weighted_laplacian(img),
    "hybrid_nuclear_laplacian":   lambda img, pxum: focus_score_hybrid_nuclear_laplacian(img),
}


def focus_score_plane(
    img2d: np.ndarray,
    metric: str = "object_based_nuclear",
    pixel_size_um: float = 0.108,
    config: Optional["PipelineConfig"] = None,
) -> float:
    """Dispatch to the requested focus metric, forwarding config tuning params when available."""
    if metric not in _FOCUS_METRICS:
        raise ValueError(f"Unsupported focus metric: {metric!r}. Choose from {list(_FOCUS_METRICS)}")
    if metric == "object_based_nuclear" and config is not None:
        return focus_score_object_based_nuclear(
            img2d, pixel_size_um=pixel_size_um,
            min_nucleus_area_um2=config.focus_min_nucleus_area_um2,
            max_nucleus_area_um2=config.focus_max_nucleus_area_um2,
            threshold_k=config.focus_threshold_k,
            sharpness_weight=config.focus_sharpness_weight,
        )
    return _FOCUS_METRICS[metric](img2d, pixel_size_um)


def get_focus_scores_for_timepoint(
    img_5d: np.ndarray,
    t: int,
    nuc_channel_idx: int,
    exclude_edge_z: int = 2,
    metric: str = "object_based_nuclear",
    pixel_size_um: float = 0.108,
    config: Optional["PipelineConfig"] = None,
) -> np.ndarray:
    """Score every Z plane for one time point; edge planes are set to NaN."""
    n_z = img_5d.shape[1]
    scores = np.full(n_z, np.nan, dtype=np.float32)
    for z in range(exclude_edge_z, n_z - exclude_edge_z):
        scores[z] = focus_score_plane(img_5d[t, z, nuc_channel_idx],
                                      metric, pixel_size_um, config=config)
    return scores


def get_best_focus_z_indices(
    img_5d: np.ndarray,
    t: int,
    nuc_channel_idx: int,
    exclude_edge_z: int = 2,
    window_radius: int = 1,
    metric: str = "object_based_nuclear",
    pixel_size_um: float = 0.108,
    config: Optional["PipelineConfig"] = None,
) -> Tuple[List[int], np.ndarray, Optional[int]]:
    """Return (keep_z, scores, best_z) for the best-focus window at time t."""
    scores = get_focus_scores_for_timepoint(
        img_5d, t, nuc_channel_idx, exclude_edge_z, metric, pixel_size_um, config=config)
    if np.all(np.isnan(scores)):
        return [], scores, None
    best_z = int(np.nanargmax(scores))
    z_start = max(exclude_edge_z, best_z - window_radius)
    z_stop  = min(len(scores) - exclude_edge_z, best_z + window_radius + 1)
    return list(range(z_start, z_stop)), scores, best_z


def plot_focus_scores_for_timepoint(
    img_5d: np.ndarray,
    t: int,
    config: PipelineConfig,
) -> Tuple:
    """Quick QC plot of focus scores across Z for one time point."""
    keep_z, scores, best_z = get_best_focus_z_indices(
        img_5d, t,
        nuc_channel_idx=config.effective_focus_channel_index,
        exclude_edge_z=config.focus_edge_z_exclusion,
        window_radius=config.focus_window_radius,
        metric=config.focus_metric,
        pixel_size_um=config.pixel_size_um,
        config=config,
    )
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(np.arange(len(scores)), scores, marker="o")
    if best_z is not None:
        ax.axvline(best_z, linestyle="--", color="tab:orange", label=f"best z={best_z}")
    for z in keep_z:
        ax.axvspan(z - 0.45, z + 0.45, alpha=0.2, color="tab:blue")
    ax.set_xlabel("Z index")
    ax.set_ylabel("Focus score")
    ax.set_title(f"Focus scores  t={t}  keep_z={keep_z}  metric={config.focus_metric}")
    ax.legend()
    plt.tight_layout()
    plt.show()
    return keep_z, scores, best_z

## Section 7 — Segmentation Helpers

In [ ]:
def load_unet_model(model_path: Union[str, Path]):
    if tf is None:
        raise ImportError("TensorFlow is not available.")
    return tf.keras.models.load_model(model_path, compile=False)


def _preprocess_patch(patch: np.ndarray) -> np.ndarray:
    """Normalise a patch to [0, 1] for model input."""
    x = patch.astype(np.float32)
    m = np.max(x)
    return x / m if m > 0 else x


def _generate_patch_starts(length: int, patch_size: int, stride: int) -> List[int]:
    """Generate patch start positions that always cover the full image edge."""
    starts = list(range(0, max(length - patch_size + 1, 1), stride)) or [0]
    last = max(0, length - patch_size)
    if starts[-1] != last:
        starts.append(last)
    return sorted(set(starts))


def extract_overlapping_patches(
    image_yxc: np.ndarray,
    patch_size: int = 512,
    stride: int = 256,
) -> Tuple[List[np.ndarray], List[Tuple[int, int]]]:
    h, w, _ = image_yxc.shape
    y_starts = _generate_patch_starts(h, patch_size, stride)
    x_starts = _generate_patch_starts(w, patch_size, stride)
    patches, coords = [], []
    for y0 in y_starts:
        for x0 in x_starts:
            patches.append(image_yxc[y0:y0 + patch_size, x0:x0 + patch_size, :])
            coords.append((y0, x0))
    return patches, coords


def _predict_batch(model, batch_patches: np.ndarray) -> np.ndarray:
    preds = model.predict(batch_patches, verbose=0)
    if preds.ndim != 4:
        raise ValueError(f"Unexpected prediction shape: {preds.shape}")
    if preds.shape[-1] == 1:
        fg = preds[..., 0]
        preds = np.stack([1.0 - fg, fg], axis=-1)
    return preds.astype(np.float32)


def stitch_probability_patches(
    patch_probs: List[np.ndarray],
    coords: List[Tuple[int, int]],
    image_shape: Tuple[int, int],
    n_classes: int,
    patch_size: int = 512,
) -> np.ndarray:
    """Average overlapping patch probabilities into a full probability map (H, W, C)."""
    h, w = image_shape
    prob_sum   = np.zeros((h, w, n_classes), dtype=np.float32)
    prob_count = np.zeros((h, w, 1),         dtype=np.float32)
    for probs, (y0, x0) in zip(patch_probs, coords):
        y1, x1 = y0 + patch_size, x0 + patch_size
        prob_sum  [y0:y1, x0:x1] += probs
        prob_count[y0:y1, x0:x1] += 1.0
    return prob_sum / np.maximum(prob_count, 1.0)


def _filter_by_area_and_circularity(
    binary_mask: np.ndarray,
    min_area_px: int = 50,
    min_circularity: float = 0.45,
) -> np.ndarray:
    labeled = measure.label(binary_mask, connectivity=2)
    out = np.zeros_like(binary_mask, dtype=bool)
    for prop in measure.regionprops(labeled):
        if prop.area < min_area_px:
            continue
        if prop.perimeter <= 0:
            continue
        if (4.0 * np.pi * prop.area) / prop.perimeter ** 2 >= min_circularity:
            out[labeled == prop.label] = True
    return out


def clean_nucleus_mask(
    nucleus_mask: np.ndarray,
    min_size_px: int = 50,
    hole_size_px: int = 200,
    opening_radius: int = 1,
    min_circularity: float = 0.45,
) -> np.ndarray:
    """Morphological cleanup: remove small objects, fill holes, opening, circularity filter."""
    mask = nucleus_mask.astype(bool)
    mask = morphology.remove_small_objects(mask, min_size=min_size_px)
    mask = morphology.remove_small_holes(mask, area_threshold=hole_size_px)
    if opening_radius > 0:
        mask = morphology.binary_opening(mask, footprint=morphology.disk(opening_radius))
    mask = ndi.binary_fill_holes(mask)
    return _filter_by_area_and_circularity(mask, min_area_px=min_size_px,
                                           min_circularity=min_circularity)


def segment_single_plane_with_overlap(
    plane_yxc: np.ndarray,
    model,
    config: PipelineConfig,
    patch_size: int = 512,
    stride: int = 256,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Segment one full plane using overlapping patches + probability stitching.

    Returns
    -------
    raw_prob_map           : (Y, X, C) raw U-Net probability — preserved for export
    class_label_map        : (Y, X) uint8 argmax class map
    nucleus_mask           : (Y, X) bool — derived from normalised probabilities
    nucleus_instance_labels: (Y, X) uint16
    norm_prob_map          : (Y, X, C) per-object normalised probability — used for thresholding
    """
    h, w, _ = plane_yxc.shape
    patches, coords = extract_overlapping_patches(plane_yxc, patch_size, stride)
    patches = [_preprocess_patch(p) for p in patches]

    patch_probs: List[np.ndarray] = []
    for i in range(0, len(patches), config.batch_size):
        batch = np.stack(patches[i:i + config.batch_size], axis=0)
        patch_probs.extend(_predict_batch(model, batch))

    n_classes = patch_probs[0].shape[-1]
    raw_prob_map = stitch_probability_patches(patch_probs, coords, (h, w), n_classes, patch_size)

    # Per-object normalisation before thresholding.
    norm_prob_map = normalize_all_class_channels(raw_prob_map, config)

    class_label_map = np.argmax(raw_prob_map, axis=-1).astype(np.uint8)

    nucleus_mask = norm_prob_map[..., config.nucleus_class_index] > config.prob_norm_final_threshold
    nucleus_mask = clean_nucleus_mask(
        nucleus_mask,
        min_size_px=config.min_nucleus_area_px,
        hole_size_px=config.nucleus_hole_size_px,
        opening_radius=config.nucleus_opening_radius_px,
        min_circularity=config.min_nucleus_circularity,
    )
    nucleus_instance_labels = measure.label(nucleus_mask, connectivity=2).astype(np.uint16)
    return raw_prob_map, class_label_map, nucleus_mask, nucleus_instance_labels, norm_prob_map

## Section 8 — Per-Object Probability Normalisation

**Rationale:** The U-Net probability maps reflect real signal variation within objects
(non-uniform fluorescence, optical gradients).  For the purpose of object isolation we
only care about *where* objects are, not how bright they are internally.  
This module rescales probability values within each detected object to the full [0, 1]
range so that a single threshold can cleanly separate object from background regardless
of internal intensity heterogeneity.

**Key design decisions:**
- Operates on the **U-Net probability output**, never the raw image
- Coarse threshold for object detection is intentionally loose; final threshold on the
  normalised map does the clean cutting
- `binary_fill_holes` before labelling prevents dim interiors fragmenting one object
  into a ring + centre as separate components
- Raw probability map is preserved separately for export and provenance
- Background channel is left untouched

In [ ]:
def normalize_probabilities_by_object(
    prob_map: np.ndarray,
    coarse_threshold: float = 0.25,
    min_object_area_px: int = 30,
) -> np.ndarray:
    """
    Normalise a 2-D single-class probability map by rescaling each detected
    object's probability values independently to [0, 1].

    Parameters
    ----------
    prob_map           : 2-D float array of per-pixel class probability (0–1)
    coarse_threshold   : loose initial threshold — finds object extents, NOT final boundary
    min_object_area_px : discard spurious specks below this area during coarse detection

    Returns
    -------
    normalised_prob : 2-D float array, same shape as prob_map
    """
    # ── Step 1: coarse binary mask to locate object extents ───────────────
    coarse_mask = prob_map > coarse_threshold

    # Fill internal holes so dim cores don't create separate components
    coarse_mask = ndi.binary_fill_holes(coarse_mask)

    # Remove tiny specks that are noise, not objects
    if min_object_area_px > 0:
        coarse_mask = morphology.remove_small_objects(coarse_mask, min_size=min_object_area_px)

    # ── Step 2: label connected components ───────────────────────────────
    labeled = measure.label(coarse_mask, connectivity=2)

    # ── Step 3: per-object min–max normalisation ──────────────────────────
    normalised_prob = np.zeros_like(prob_map, dtype=np.float32)

    for prop in measure.regionprops(labeled):
        obj_mask = labeled == prop.label
        obj_probs = prob_map[obj_mask]
        p_min, p_max = float(obj_probs.min()), float(obj_probs.max())

        if p_max > p_min:
            normalised_prob[obj_mask] = (obj_probs - p_min) / (p_max - p_min)
        else:
            # Uniform region — treat as fully confident
            normalised_prob[obj_mask] = 1.0

    return normalised_prob


def normalize_all_class_channels(
    class_prob_map: np.ndarray,
    config: PipelineConfig,
) -> np.ndarray:
    """
    Apply per-object probability normalisation to selected class channels
    of a full (Y, X, C) probability map.

    Background channel (index 0) is left untouched.  Nucleus and droplet
    channels are normalised independently based on config flags.

    Returns a new (Y, X, C) array — input is not modified.
    """
    normalised = class_prob_map.copy().astype(np.float32)

    if config.prob_norm_nucleus:
        normalised[..., config.nucleus_class_index] = normalize_probabilities_by_object(
            class_prob_map[..., config.nucleus_class_index],
            coarse_threshold=config.prob_norm_coarse_threshold,
            min_object_area_px=config.prob_norm_min_object_area_px,
        )

    if config.prob_norm_droplet:
        normalised[..., config.droplet_class_index] = normalize_probabilities_by_object(
            class_prob_map[..., config.droplet_class_index],
            coarse_threshold=config.prob_norm_coarse_threshold,
            min_object_area_px=config.prob_norm_min_object_area_px,
        )

    return normalised


def process_probability_hyperstack_normalised(
    prob_hyperstack: np.ndarray,
    config: PipelineConfig,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Apply per-object normalisation across a full (T, Z, C, Y, X) probability
    hyperstack and return both the normalised map and the resulting binary mask.

    Useful for post-hoc reprocessing of a saved raw probability TIFF without
    re-running inference.

    Returns
    -------
    normalised_hyperstack : float32 (T, Z, C, Y, X)
    binary_hyperstack     : uint8   (T, Z, Y, X)  — nucleus class only
    """
    T, Z, C, Y, X = prob_hyperstack.shape
    normalised_hyperstack = np.zeros_like(prob_hyperstack, dtype=np.float32)
    binary_hyperstack     = np.zeros((T, Z, Y, X), dtype=np.uint8)

    for t in range(T):
        for z in range(Z):
            # prob_hyperstack is (T, Z, C, Y, X) — need (Y, X, C) for the normaliser
            plane_yxc = np.moveaxis(prob_hyperstack[t, z], 0, -1)
            norm_yxc  = normalize_all_class_channels(plane_yxc, config)
            normalised_hyperstack[t, z] = np.moveaxis(norm_yxc, -1, 0)

            nuc_norm = norm_yxc[..., config.nucleus_class_index]
            binary_hyperstack[t, z] = (
                (nuc_norm > config.prob_norm_final_threshold).astype(np.uint8) * 255
            )

    return normalised_hyperstack, binary_hyperstack

## Section 9 — TIFF Save Helpers

In [ ]:
def _write_hyperstack_tiff(
    arr: np.ndarray,
    out_path: Path,
    axes: str,
    prefer_imagej: bool = True,
) -> None:
    if tiff is None:
        raise ImportError("tifffile is required.")
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    arr = np.asarray(arr)
    # BigTIFF for arrays >= ~2 GB to avoid ImageJ truncation
    use_imagej = prefer_imagej and arr.nbytes < (2 * 1024**3 - 32 * 1024**2)
    tiff.imwrite(out_path, arr,
                 imagej=use_imagej, bigtiff=not use_imagej,
                 metadata={"axes": axes})
    print(f"Saved: {out_path}  shape={arr.shape}  dtype={arr.dtype}")


def save_probability_hyperstack_tiff(probability_5d: np.ndarray, out_path: Path) -> None:
    """Save raw U-Net probability output (T, Z, C, Y, X) as float16 BigTIFF."""
    _write_hyperstack_tiff(probability_5d.astype(np.float16), out_path,
                           axes="TZCYX", prefer_imagej=False)


def save_normalised_probability_hyperstack_tiff(norm_5d: np.ndarray, out_path: Path) -> None:
    """Save per-object normalised probability (T, Z, C, Y, X) as float16 BigTIFF."""
    _write_hyperstack_tiff(norm_5d.astype(np.float16), out_path,
                           axes="TZCYX", prefer_imagej=False)


def save_class_hyperstack_tiff(class_mask_5d: np.ndarray, out_path: Path) -> None:
    _write_hyperstack_tiff((class_mask_5d.astype(np.uint8) * 255), out_path,
                           axes="TZCYX", prefer_imagej=True)


def save_label_hyperstack_tiff(label_4d: np.ndarray, out_path: Path) -> None:
    _write_hyperstack_tiff(label_4d.astype(np.uint8), out_path,
                           axes="TZYX", prefer_imagej=True)


def save_instance_hyperstack_tiff(instance_4d: np.ndarray, out_path: Path) -> None:
    _write_hyperstack_tiff(instance_4d, out_path, axes="TZYX", prefer_imagej=True)


def create_probability_memmap(
    shape: Tuple[int, int, int, int, int],
    out_dir: Path,
    dtype: np.dtype = np.float16,
    tag: str = "raw",
) -> Tuple[np.memmap, Path]:
    out_dir.mkdir(parents=True, exist_ok=True)
    temp_path = out_dir / f"probability_hyperstack_{tag}_temp.dat"
    if temp_path.exists():
        temp_path.unlink()
    return np.memmap(temp_path, mode="w+", dtype=dtype, shape=shape), temp_path

## Section 10 — Regionprops Helper

In [ ]:
def regionprops_to_rows(
    labeled_mask: np.ndarray,
    t_idx: int,
    z_idx: int,
    class_id: int,
    class_name: str,
    min_area_px: int = 0,
) -> List[Dict]:
    """
    Convert a labelled mask to a list of row dicts suitable for DataFrame construction.

    Centroids, area, perimeter, circularity, and bounding box are recorded.
    Physical-unit conversions are applied downstream via cfg.pixel_size_um.
    """
    rows = []
    for prop in measure.regionprops(labeled_mask):
        if prop.area < min_area_px:
            continue
        cy, cx = prop.centroid
        perimeter = float(prop.perimeter)
        circularity = (
            (4.0 * np.pi * prop.area) / perimeter ** 2
            if perimeter > 0 else np.nan
        )
        rows.append({
            "t": t_idx, "z": z_idx,
            "class_id": class_id, "class_name": class_name,
            "label": int(prop.label),
            "centroid_x_px": float(cx), "centroid_y_px": float(cy),
            "area_px": int(prop.area),
            "perimeter_px": perimeter,
            "circularity": circularity,
            "bbox_min_row": int(prop.bbox[0]), "bbox_min_col": int(prop.bbox[1]),
            "bbox_max_row": int(prop.bbox[2]), "bbox_max_col": int(prop.bbox[3]),
        })
    return rows

## Section 11 — Full Segmentation Loop

In [ ]:
def run_segmentation_for_all_planes(
    img_5d: np.ndarray,
    model,
    config: PipelineConfig,
    save_binary_masks: bool = True,
    save_probability_tiff: bool = True,
) -> pd.DataFrame:
    """
    Run U-Net segmentation over the full (T, Z, C, Y, X) hyperstack.

    Per-object probability normalisation is applied plane-by-plane inside
    segment_single_plane_with_overlap so that binary mask generation is uniform
    regardless of internal fluorescence variation.

    Both raw and normalised probability hyperstacks are written to disk for
    provenance and QC.

    When config.use_focus_z_selection is True, only best_z ± focus_window_radius
    planes are segmented per time point.  Skipped planes are recorded in the
    index with included=False so downstream file-based steps remain robust.

    Returns
    -------
    seg_index_df : DataFrame recording every plane with inclusion status,
                   focus score, and path to the saved binary mask.
    """
    T, Z = img_5d.shape[0], img_5d.shape[1]
    Y, X = img_5d.shape[-2], img_5d.shape[-1]

    records:  List[dict] = []
    roi_rows: List[dict] = []

    class_mask_5d       = np.zeros((T, Z, 3, Y, X), dtype=np.uint8)
    class_label_4d      = np.zeros((T, Z, Y, X),    dtype=np.uint8)
    nucleus_instance_4d = np.zeros((T, Z, Y, X),    dtype=np.uint16)
    droplet_instance_4d = np.zeros((T, Z, Y, X),    dtype=np.uint16)

    # Memmaps for both raw and normalised probability hyperstacks
    raw_prob_5d = raw_prob_temp = None
    nrm_prob_5d = nrm_prob_temp = None

    if save_probability_tiff:
        raw_prob_5d, raw_prob_temp = create_probability_memmap(
            shape=(T, Z, 3, Y, X), out_dir=config.seg_dir, tag="raw")
        nrm_prob_5d, nrm_prob_temp = create_probability_memmap(
            shape=(T, Z, 3, Y, X), out_dir=config.seg_dir, tag="normalised")

    try:
        for t_idx in range(T):
            if config.use_focus_z_selection:
                keep_z, focus_scores, best_z = get_best_focus_z_indices(
                    img_5d=img_5d, t=t_idx,
                    nuc_channel_idx=config.effective_focus_channel_index,
                    exclude_edge_z=config.focus_edge_z_exclusion,
                    window_radius=config.focus_window_radius,
                    metric=config.focus_metric,
                    pixel_size_um=config.pixel_size_um,
                    config=config,
                )
                print(f"t={t_idx}: best focus z={best_z}, segmenting z={keep_z}")
            else:
                focus_scores = np.full(Z, np.nan, dtype=np.float32)
                best_z, keep_z = None, list(range(Z))

            keep_z_set = set(keep_z)

            for z_idx in range(Z):
                fscore = (
                    float(focus_scores[z_idx])
                    if z_idx < len(focus_scores) and np.isfinite(focus_scores[z_idx])
                    else np.nan
                )
                out_path = config.seg_dir / f"nuclear_mask_t{t_idx:03d}_z{z_idx:03d}.npy"

                if z_idx not in keep_z_set:
                    if z_idx < config.focus_edge_z_exclusion or z_idx >= Z - config.focus_edge_z_exclusion:
                        reason = "edge_z_excluded"
                    else:
                        reason = "outside_best_focus_window" if config.use_focus_z_selection else "skipped"
                    if save_binary_masks:
                        np.save(out_path, np.zeros((Y, X), dtype=np.uint8))
                    records.append({
                        "t": t_idx, "z": z_idx, "included": False,
                        "reason": reason, "focus_score": fscore,
                        "best_focus_z": best_z, "mask_path": str(out_path),
                    })
                    continue

                plane_yxc = get_full_plane_yxc(img_5d, t_idx, z_idx)

                # Returns raw prob, class map, nucleus mask, instances, normalised prob
                raw_prob_map, class_label_map, nucleus_mask, nucleus_instance_labels, norm_prob_map = \
                    segment_single_plane_with_overlap(
                        plane_yxc, model, config,
                        patch_size=config.patch_size, stride=config.patch_stride)

                background_mask   = class_label_map == config.background_class_index
                droplet_mask      = class_label_map == config.droplet_class_index
                droplet_instances = measure.label(droplet_mask, connectivity=2).astype(np.uint16)

                if save_probability_tiff:
                    if raw_prob_5d is not None:
                        raw_prob_5d[t_idx, z_idx] = np.moveaxis(
                            raw_prob_map.astype(np.float16), -1, 0)
                    if nrm_prob_5d is not None:
                        nrm_prob_5d[t_idx, z_idx] = np.moveaxis(
                            norm_prob_map.astype(np.float16), -1, 0)

                class_mask_5d      [t_idx, z_idx, 0] = background_mask.astype(np.uint8)
                class_mask_5d      [t_idx, z_idx, 1] = droplet_mask.astype(np.uint8)
                class_mask_5d      [t_idx, z_idx, 2] = nucleus_mask.astype(np.uint8)
                class_label_4d     [t_idx, z_idx]    = class_label_map
                nucleus_instance_4d[t_idx, z_idx]    = nucleus_instance_labels
                droplet_instance_4d[t_idx, z_idx]    = droplet_instances

                roi_rows.extend(regionprops_to_rows(
                    nucleus_instance_labels, t_idx, z_idx,
                    config.nucleus_class_index, "nucleus"))
                roi_rows.extend(regionprops_to_rows(
                    droplet_instances, t_idx, z_idx,
                    config.droplet_class_index, "droplet"))

                if save_binary_masks:
                    np.save(out_path, nucleus_mask.astype(np.uint8))

                records.append({
                    "t": t_idx, "z": z_idx, "included": True,
                    "reason": "segmented", "focus_score": fscore,
                    "best_focus_z": best_z, "mask_path": str(out_path),
                    "nucleus_pixels": int(nucleus_mask.sum()),
                    "droplet_pixels": int(droplet_mask.sum()),
                })

                del raw_prob_map, norm_prob_map, class_label_map
                del nucleus_mask, nucleus_instance_labels
                del droplet_mask, background_mask, droplet_instances, plane_yxc
                gc.collect()

        # ── Write hyperstacks ─────────────────────────────────────────────
        save_class_hyperstack_tiff(class_mask_5d, config.segmentation_class_hyperstack_path)
        save_label_hyperstack_tiff(class_label_4d, config.segmentation_label_hyperstack_path)
        save_instance_hyperstack_tiff(nucleus_instance_4d, config.nucleus_instance_hyperstack_path)
        save_instance_hyperstack_tiff(droplet_instance_4d, config.droplet_instance_hyperstack_path)

        if save_probability_tiff:
            if raw_prob_5d is not None:
                raw_prob_5d.flush()
                save_probability_hyperstack_tiff(
                    raw_prob_5d, config.segmentation_probability_hyperstack_path)
            if nrm_prob_5d is not None:
                nrm_prob_5d.flush()
                save_normalised_probability_hyperstack_tiff(
                    nrm_prob_5d, config.normalised_probability_hyperstack_path)

        pd.DataFrame(roi_rows).to_pickle(config.segmentation_roi_table_path)
        seg_index_df = pd.DataFrame(records)
        seg_index_df.to_pickle(config.segmentation_index_path)
        return seg_index_df

    finally:
        for mm, tmp in [(raw_prob_5d, raw_prob_temp), (nrm_prob_5d, nrm_prob_temp)]:
            if isinstance(mm, np.memmap):
                try:
                    mm.flush()
                except Exception:
                    pass
            if tmp is not None and Path(tmp).exists():
                try:
                    Path(tmp).unlink()
                except Exception:
                    pass

## Section 12 — Object Extraction from Saved Masks

In [ ]:
def extract_objects_from_saved_masks(
    seg_index_df: pd.DataFrame,
    config: PipelineConfig,
) -> pd.DataFrame:
    """
    Load saved binary mask files and extract region properties.
    Rows with included=False are skipped — they hold empty masks.
    """
    included = seg_index_df[seg_index_df["included"].astype(bool)]
    all_rows: List[dict] = []

    for row in included.itertuples(index=False):
        mask = np.load(row.mask_path).astype(bool)
        rows = regionprops_to_rows(
            measure.label(mask, connectivity=2),
            t_idx=row.t, z_idx=row.z,
            class_id=config.nucleus_class_index, class_name="nucleus",
            min_area_px=config.min_nucleus_area_px,
        )
        all_rows.extend(rows)

    objects_df = pd.DataFrame(all_rows) if all_rows else pd.DataFrame()
    objects_df.to_pickle(config.obj_dir / "plane_objects.pkl")
    return objects_df

## Section 13 — Distance-Based Helpers

In [ ]:
def nearest_neighbor_matches(
    source_df: pd.DataFrame,
    target_df: pd.DataFrame,
    max_dist_px: float,
) -> List[Tuple[int, int, float]]:
    """Return greedy nearest-neighbour matches within max_dist_px (one-to-one)."""
    if source_df.empty or target_df.empty:
        return []
    src_xy = source_df[["centroid_x_px", "centroid_y_px"]].to_numpy()
    tgt_xy = target_df[["centroid_x_px", "centroid_y_px"]].to_numpy()
    tree = cKDTree(tgt_xy)
    dists, idxs = tree.query(src_xy, distance_upper_bound=max_dist_px)

    matches, used = [], set()
    for i in np.argsort(dists):
        dist, j = dists[i], idxs[i]
        if np.isinf(dist) or j in used:
            continue
        matches.append((int(i), int(j), float(dist)))
        used.add(int(j))
    return matches

## Section 14 — Group Nuclei Across Z

In [ ]:
def group_nuclei_across_z(objects_df: pd.DataFrame, config: PipelineConfig) -> pd.DataFrame:
    """
    Link nucleus detections across consecutive Z planes within each time point
    using nearest-neighbour matching.  Assigns a nucleus_3d_id to each detection.
    """
    if objects_df.empty:
        return pd.DataFrame()

    grouped_rows = []
    next_group_id = 1

    for t_idx, df_t in objects_df.groupby("t", sort=True):
        df_t = df_t.sort_values(["z", "label"]).reset_index(drop=True).copy()
        df_t["nucleus_3d_id"] = -1
        z_values = sorted(df_t["z"].unique())

        for idx in df_t.index[df_t["z"] == z_values[0]]:
            df_t.at[idx, "nucleus_3d_id"] = next_group_id
            next_group_id += 1

        for z_prev, z_curr in zip(z_values[:-1], z_values[1:]):
            prev_df = df_t[df_t["z"] == z_prev].reset_index()
            curr_df = df_t[df_t["z"] == z_curr].reset_index()
            matched_curr = set()

            for i_prev, i_curr, _ in nearest_neighbor_matches(
                    prev_df, curr_df, config.z_group_tolerance_px):
                prev_global = int(prev_df.loc[i_prev, "index"])
                curr_global = int(curr_df.loc[i_curr, "index"])
                matched_curr.add(curr_global)

                grp = df_t.at[prev_global, "nucleus_3d_id"]
                if grp == -1:
                    grp = next_group_id
                    df_t.at[prev_global, "nucleus_3d_id"] = grp
                    next_group_id += 1
                df_t.at[curr_global, "nucleus_3d_id"] = grp

            for curr_global in curr_df["index"].tolist():
                if curr_global not in matched_curr and df_t.at[curr_global, "nucleus_3d_id"] == -1:
                    df_t.at[curr_global, "nucleus_3d_id"] = next_group_id
                    next_group_id += 1

        grouped_rows.append(df_t)

    grouped_z_df = pd.concat(grouped_rows, ignore_index=True)
    grouped_z_df.to_pickle(config.obj_dir / "grouped_z_objects.pkl")
    return grouped_z_df

## Section 15 — Select Best Z Per Nucleus

In [ ]:
def select_best_z_per_nucleus(grouped_z_df: pd.DataFrame, config: PipelineConfig) -> pd.DataFrame:
    """
    For each nucleus_3d_id within each time point, retain the single Z plane
    with the largest cross-sectional area (proxy for equatorial section).
    """
    if grouped_z_df.empty:
        return pd.DataFrame()
    best_z_df = (
        grouped_z_df
        .sort_values(["t", "nucleus_3d_id", "area_px"], ascending=[True, True, False])
        .groupby(["t", "nucleus_3d_id"], as_index=False)
        .head(1)
        .reset_index(drop=True)
    )
    best_z_df.to_pickle(config.obj_dir / "best_z_nuclei.pkl")
    return best_z_df

## Section 16 — Flag Close / Multi-Nucleus Droplets

In [ ]:
def flag_close_nuclei(best_z_df: pd.DataFrame, config: PipelineConfig) -> pd.DataFrame:
    """
    Flag nucleus detections that are within multi_nucleus_exclusion_px of
    any other nucleus at the same time point.  Flagged nuclei are marked
    valid_single_nucleus=False and excluded from tracking.
    """
    if best_z_df.empty:
        return pd.DataFrame()
    out = []
    for _, df_t in best_z_df.groupby("t", sort=True):
        df_t = df_t.copy().reset_index(drop=True)
        coords = df_t[["centroid_x_px", "centroid_y_px"]].to_numpy()
        tree = cKDTree(coords)
        close_flags = np.zeros(len(df_t), dtype=bool)
        for i, xy in enumerate(coords):
            neighbors = [j for j in tree.query_ball_point(xy, r=config.multi_nucleus_exclusion_px) if j != i]
            if neighbors:
                close_flags[i] = True
                for j in neighbors:
                    close_flags[j] = True
        df_t["valid_single_nucleus"] = ~close_flags
        out.append(df_t)
    filtered_df = pd.concat(out, ignore_index=True)
    filtered_df.to_pickle(config.obj_dir / "best_z_nuclei_with_exclusion.pkl")
    return filtered_df

## Section 17 — Tile Assignment and True Acquisition Time

In [ ]:
def assign_tile_index_from_xy(
    x_px: float, y_px: float,
    image_width_px: int, image_height_px: int,
    config: PipelineConfig,
) -> Tuple[int, int, int]:
    col = min(int(x_px // (image_width_px  / config.tile_cols)), config.tile_cols - 1)
    row = min(int(y_px // (image_height_px / config.tile_rows)), config.tile_rows - 1)
    if config.serpentine_scan and row % 2 != 0:
        tile_index = row * config.tile_cols + (config.tile_cols - 1 - col)
    else:
        tile_index = row * config.tile_cols + col
    return row, col, tile_index


def add_tile_timing_metadata(
    nuclei_df: pd.DataFrame,
    img_5d: np.ndarray,
    config: PipelineConfig,
) -> pd.DataFrame:
    """Add tile_row, tile_col, tile_index, tile_offset_min, and true_time_min to each nucleus."""
    if nuclei_df.empty:
        return pd.DataFrame()
    image_height_px, image_width_px = img_5d.shape[-2], img_5d.shape[-1]
    out_rows = []
    tiles_per_frame = config.tile_rows * config.tile_cols * config.minutes_per_tile
    for row in nuclei_df.itertuples(index=False):
        tile_row, tile_col, tile_index = assign_tile_index_from_xy(
            row.centroid_x_px, row.centroid_y_px,
            image_width_px, image_height_px, config)
        d = row._asdict()
        d["tile_row"]        = tile_row
        d["tile_col"]        = tile_col
        d["tile_index"]      = tile_index
        d["tile_offset_min"] = tile_index * config.minutes_per_tile
        d["true_time_min"]   = row.t * tiles_per_frame + d["tile_offset_min"]
        out_rows.append(d)
    timed_df = pd.DataFrame(out_rows)
    timed_df.to_pickle(config.track_dir / "best_z_nuclei_timed.pkl")
    return timed_df

## Section 18 — Hybrid DBSCAN Tracker

1. Build local spatial neighbourhoods with DBSCAN from consecutive frames.
2. Score candidate links by distance + area consistency.
3. One-to-one Hungarian assignment inside each local neighbourhood.
4. Reject crowded or low-confidence links rather than forcing a track.

In [ ]:
def build_dbscan_candidate_clusters(
    prev_df: pd.DataFrame,
    curr_df: pd.DataFrame,
    config: PipelineConfig,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if DBSCAN is None:
        raise ImportError("scikit-learn is required for DBSCAN-assisted tracking.")

    prev_local = prev_df.copy().reset_index(drop=True)
    curr_local = curr_df.copy().reset_index(drop=True)
    prev_local["frame_role"] = "prev"
    curr_local["frame_role"] = "curr"
    prev_local["frame_local_index"] = np.arange(len(prev_local))
    curr_local["frame_local_index"] = np.arange(len(curr_local))

    pooled = pd.concat([prev_local, curr_local], ignore_index=True)
    if pooled.empty:
        for col in ("dbscan_cluster_id", "dbscan_cluster_size", "dbscan_is_crowded"):
            pooled[col] = pd.Series(dtype=int if col != "dbscan_is_crowded" else bool)
        return prev_local, curr_local, pooled

    coords = pooled[["centroid_x_px", "centroid_y_px"]].to_numpy()
    labels = DBSCAN(eps=config.track_dbscan_eps_px,
                    min_samples=config.track_dbscan_min_samples).fit_predict(coords)
    pooled["dbscan_cluster_id"] = labels

    crowded_flags, cluster_sizes = [], []
    for _, df_c in pooled.groupby("dbscan_cluster_id", sort=False):
        n_prev = int((df_c["frame_role"] == "prev").sum())
        n_curr = int((df_c["frame_role"] == "curr").sum())
        is_crowded = n_prev > 1 or n_curr > 1
        crowded_flags.extend([is_crowded] * len(df_c))
        cluster_sizes.extend([len(df_c)] * len(df_c))
    pooled["dbscan_cluster_size"] = cluster_sizes
    pooled["dbscan_is_crowded"]   = crowded_flags

    prev_out = pooled[pooled["frame_role"] == "prev"].copy().reset_index(drop=True)
    curr_out = pooled[pooled["frame_role"] == "curr"].copy().reset_index(drop=True)
    return prev_out, curr_out, pooled


def match_cluster_with_assignment(
    prev_cluster: pd.DataFrame,
    curr_cluster: pd.DataFrame,
    config: PipelineConfig,
) -> List[dict]:
    if prev_cluster.empty or curr_cluster.empty:
        return []

    n_prev, n_curr = len(prev_cluster), len(curr_cluster)
    cost_matrix = np.full((n_prev, n_curr), np.nan, dtype=float)
    max_dist_base = float(config.time_track_tolerance_px)

    for i, prev_row in prev_cluster.iterrows():
        for j, curr_row in curr_cluster.iterrows():
            dx = float(curr_row["centroid_x_px"]) - float(prev_row["centroid_x_px"])
            dy = float(curr_row["centroid_y_px"]) - float(prev_row["centroid_y_px"])
            dist_px = math.hypot(dx, dy)

            max_dist = max_dist_base
            if prev_row.get("dbscan_is_crowded", False) or curr_row.get("dbscan_is_crowded", False):
                max_dist *= config.track_crowded_distance_scale

            prev_area = max(float(prev_row["area_px"]), 1.0)
            curr_area = max(float(curr_row["area_px"]), 1.0)
            area_ratio = max(prev_area, curr_area) / min(prev_area, curr_area)

            if dist_px <= max_dist and area_ratio <= config.track_area_ratio_max:
                area_cost = abs(math.log(curr_area / prev_area))
                dist_cost = dist_px / max(max_dist, 1e-6)
                cost_matrix[i, j] = dist_cost + config.track_area_log_weight * area_cost

    finite_mask = np.isfinite(cost_matrix)
    valid_rows  = np.where(finite_mask.any(axis=1))[0]
    valid_cols  = np.where(finite_mask.any(axis=0))[0]
    if len(valid_rows) == 0 or len(valid_cols) == 0:
        return []

    large_penalty = 1e9
    reduced = np.where(
        np.isfinite(cost_matrix[np.ix_(valid_rows, valid_cols)]),
        cost_matrix[np.ix_(valid_rows, valid_cols)], large_penalty)

    bad_rows = np.all(reduced >= large_penalty, axis=1)
    bad_cols = np.all(reduced >= large_penalty, axis=0)
    if bad_rows.any() or bad_cols.any():
        keep_r = np.where(~bad_rows)[0]
        keep_c = np.where(~bad_cols)[0]
        if len(keep_r) == 0 or len(keep_c) == 0:
            return []
        reduced    = reduced   [np.ix_(keep_r, keep_c)]
        valid_rows = valid_rows[keep_r]
        valid_cols = valid_cols[keep_c]

    row_ind, col_ind = linear_sum_assignment(reduced)

    matches = []
    for r, c in zip(row_ind, col_ind):
        orig_i = valid_rows[r]
        orig_j = valid_cols[c]
        link_cost = cost_matrix[orig_i, orig_j]
        if not np.isfinite(link_cost):
            continue
        pr = prev_cluster.iloc[orig_i]
        cr = curr_cluster.iloc[orig_j]
        dist_px = math.hypot(
            float(cr["centroid_x_px"]) - float(pr["centroid_x_px"]),
            float(cr["centroid_y_px"]) - float(pr["centroid_y_px"]),
        )
        matches.append({
            "prev_index":          int(pr["global_index"]),
            "curr_index":          int(cr["global_index"]),
            "link_cost":           float(link_cost),
            "link_distance_px":    float(dist_px),
            "dbscan_cluster_id":   int(pr.get("dbscan_cluster_id", -1)),
            "dbscan_cluster_size": int(pr.get("dbscan_cluster_size", 0)),
            "dbscan_is_crowded":   bool(pr.get("dbscan_is_crowded", False)),
        })
    return matches


def assign_track_ids_hybrid_dbscan(
    timed_df: pd.DataFrame,
    config: PipelineConfig,
    valid_only: bool = False,
) -> pd.DataFrame:
    """Assign persistent track IDs using hybrid DBSCAN + Hungarian assignment."""
    if timed_df.empty:
        return pd.DataFrame()

    df = timed_df.copy().reset_index(drop=True)
    df["global_index"] = np.arange(len(df), dtype=int)
    df = df.sort_values(["t", "nucleus_3d_id"]).reset_index(drop=True)

    df["track_id"]                  = -1
    df["track_link_distance_px"]    = np.nan
    df["track_link_cost"]           = np.nan
    df["track_dbscan_cluster_id"]   = -1
    df["track_dbscan_cluster_size"] = 0
    df["track_dbscan_is_crowded"]   = False
    df["track_link_method"]         = "unassigned"

    time_values = sorted(df["t"].unique())
    if not time_values:
        return df

    next_track_id = 1
    for idx in df.index[df["t"] == time_values[0]]:
        df.at[idx, "track_id"]          = next_track_id
        df.at[idx, "track_link_method"] = "seed"
        next_track_id += 1

    for t_prev, t_curr in zip(time_values[:-1], time_values[1:]):
        prev_df = df[df["t"] == t_prev].reset_index()
        curr_df = df[df["t"] == t_curr].reset_index()

        prev_clustered, curr_clustered, pooled = build_dbscan_candidate_clusters(
            prev_df, curr_df, config)

        all_matches: List[dict] = []
        for cluster_id in sorted(pooled["dbscan_cluster_id"].unique()):
            pc = prev_clustered[prev_clustered["dbscan_cluster_id"] == cluster_id].reset_index(drop=True)
            cc = curr_clustered[curr_clustered["dbscan_cluster_id"] == cluster_id].reset_index(drop=True)
            all_matches.extend(match_cluster_with_assignment(pc, cc, config))

        matched_curr_global: set = set()

        for match in sorted(all_matches, key=lambda x: x["link_cost"]):
            prev_global = int(match["prev_index"])
            curr_global = int(match["curr_index"])

            if curr_global in matched_curr_global:
                continue

            track_id = int(df.at[prev_global, "track_id"])
            if track_id == -1:
                track_id = next_track_id
                df.at[prev_global, "track_id"]          = track_id
                df.at[prev_global, "track_link_method"] = "backfilled_seed"
                next_track_id += 1

            matched_curr_global.add(curr_global)
            df.at[curr_global, "track_id"]                  = track_id
            df.at[curr_global, "track_link_distance_px"]    = float(match["link_distance_px"])
            df.at[curr_global, "track_link_cost"]           = float(match["link_cost"])
            df.at[curr_global, "track_dbscan_cluster_id"]   = int(match["dbscan_cluster_id"])
            df.at[curr_global, "track_dbscan_cluster_size"] = int(match["dbscan_cluster_size"])
            df.at[curr_global, "track_dbscan_is_crowded"]   = bool(match["dbscan_is_crowded"])
            df.at[curr_global, "track_link_method"]         = "hybrid_dbscan"

        for _, row in curr_clustered.iterrows():
            curr_global = int(row["index"])
            df.at[curr_global, "track_dbscan_cluster_id"]   = int(row["dbscan_cluster_id"])
            df.at[curr_global, "track_dbscan_cluster_size"] = int(row["dbscan_cluster_size"])
            df.at[curr_global, "track_dbscan_is_crowded"]   = bool(row["dbscan_is_crowded"])
            if df.at[curr_global, "track_id"] == -1:
                df.at[curr_global, "track_id"]          = next_track_id
                df.at[curr_global, "track_link_method"] = "new_track_unmatched"
                next_track_id += 1

    df.to_pickle(config.track_dir / "tracked_nuclei.pkl")
    return df


def summarize_tracking_debug(tracked_df: pd.DataFrame) -> pd.DataFrame:
    if tracked_df.empty:
        return pd.DataFrame()
    track_len = tracked_df.groupby("track_id").size()
    return pd.DataFrame([{
        "n_rows":                 int(len(tracked_df)),
        "n_tracks":               int(tracked_df["track_id"].nunique()),
        "median_track_length":    float(track_len.median()),
        "mean_track_length":      float(track_len.mean()),
        "fraction_crowded_links": float(tracked_df["track_dbscan_is_crowded"].fillna(False).mean()),
        "fraction_new_tracks":    float((tracked_df["track_link_method"] == "new_track_unmatched").mean()),
    }])

## Section 19 — Halo / Ring / N:C Ratio Analysis

In [ ]:
def build_cumulative_halo_masks(
    nucleus_mask: np.ndarray,
    config: PipelineConfig,
) -> Dict[str, np.ndarray]:
    """Build concentric halo, ring, and cytoplasm masks from a binary nucleus mask."""
    dist_outside = ndi.distance_transform_edt(~nucleus_mask)
    masks = {"nucleus_mask": nucleus_mask.astype(bool)}

    for i in range(1, config.n_halos + 1):
        masks[f"halo_{i}_cum"] = dist_outside <= i * config.halo_step_px

    for i in range(1, config.n_halos + 1):
        prev = nucleus_mask if i == 1 else masks[f"halo_{i-1}_cum"]
        masks[f"ring_{i}"] = masks[f"halo_{i}_cum"] & ~prev

    masks["nucleus_eroded"] = (
        ndi.binary_erosion(nucleus_mask, structure=morphology.disk(config.erosion_px))
        if config.erosion_px > 0 else nucleus_mask.copy()
    )
    masks["cytoplasm_mask"] = masks[f"halo_{config.n_halos}_cum"] & ~nucleus_mask
    return masks


def _mask_stats(intensity_image: np.ndarray, mask: np.ndarray) -> Dict[str, float]:
    area_px = int(mask.sum())
    if area_px == 0:
        return {"area_px": 0, "intden": 0.0, "mean_intensity": np.nan}
    vals = intensity_image[mask]
    return {"area_px": area_px, "intden": float(vals.sum()), "mean_intensity": float(vals.mean())}


def measure_fiji_equivalent_halos(
    intensity_image: np.ndarray,
    nucleus_mask: np.ndarray,
    config: PipelineConfig,
) -> Dict[str, float]:
    """
    Measure intensity within the nucleus, eroded nucleus, cytoplasm halo, and
    each concentric ring.  Measurement is performed on the ORIGINAL unmodified
    intensity image — masks are used only to define ROIs.
    """
    masks = build_cumulative_halo_masks(nucleus_mask, config)
    out: Dict[str, float] = {}

    for key in ("nucleus_mask", "nucleus_eroded", "cytoplasm_mask"):
        prefix = key.replace("_mask", "").replace("_cum", "")
        for k, v in _mask_stats(intensity_image, masks[key]).items():
            out[f"{prefix}_{k}"] = v

    for i in range(1, config.n_halos + 1):
        for k, v in _mask_stats(intensity_image, masks[f"halo_{i}_cum"]).items():
            out[f"halo_{i}_cum_{k}"] = v
        for k, v in _mask_stats(intensity_image, masks[f"ring_{i}"]).items():
            out[f"ring_{i}_{k}"] = v

    nm = out.get("nucleus_mean_intensity", np.nan)
    cm = out.get("cytoplasm_mean_intensity", np.nan)
    out["nc_ratio"]          = float(nm / cm) if np.isfinite(nm) and np.isfinite(cm) and cm != 0 else np.nan
    out["nc_ratio_fraction"] = float(nm / (nm + cm)) if np.isfinite(nm) and np.isfinite(cm) and (nm + cm) != 0 else np.nan
    return out


def recover_nucleus_mask_from_plane(
    t_idx: int, z_idx: int,
    centroid_x_px: float, centroid_y_px: float,
    config: PipelineConfig,
) -> np.ndarray:
    """Load a saved binary mask and return only the component nearest the given centroid."""
    mask_path = config.seg_dir / f"nuclear_mask_t{t_idx:03d}_z{z_idx:03d}.npy"
    mask = np.load(mask_path).astype(bool)
    labeled = measure.label(mask, connectivity=2)
    best_label, best_dist = None, np.inf
    for prop in measure.regionprops(labeled):
        cy, cx = prop.centroid
        dist = math.hypot(cx - centroid_x_px, cy - centroid_y_px)
        if dist < best_dist:
            best_dist, best_label = dist, prop.label
    return (labeled == best_label) if best_label is not None else np.zeros_like(mask, dtype=bool)


def _process_single_tracked_nucleus(
    row,
    img_5d: np.ndarray,
    config: PipelineConfig,
) -> dict:
    """Worker function: recover mask, measure on original image, append halo metrics."""
    nucleus_mask  = recover_nucleus_mask_from_plane(
        int(row.t), int(row.z), float(row.centroid_x_px), float(row.centroid_y_px), config)
    nuclear_plane = get_nuclear_plane(img_5d, int(row.t), int(row.z), config)
    halo_metrics  = measure_fiji_equivalent_halos(nuclear_plane, nucleus_mask, config)
    rec = row._asdict()
    rec.update(halo_metrics)
    rec["nucleus_area_um2"]   = rec.get("nucleus_area_px",   0) * config.pixel_size_um ** 2
    rec["cytoplasm_area_um2"] = rec.get("cytoplasm_area_px", 0) * config.pixel_size_um ** 2
    return rec


def run_halo_analysis_for_tracked_nuclei(
    tracked_df: pd.DataFrame,
    img_5d: np.ndarray,
    config: PipelineConfig,
) -> pd.DataFrame:
    """Parallelised halo analysis across all tracked nuclei."""
    if tracked_df.empty:
        halo_df = pd.DataFrame()
        halo_df.to_pickle(config.analysis_dir / "halo_analysis.pkl")
        return halo_df
    rows = Parallel(
        n_jobs=max(1, int(config.n_workers)), backend="threading", batch_size=1
    )(
        delayed(_process_single_tracked_nucleus)(row, img_5d, config)
        for row in tracked_df.itertuples(index=False)
    )
    halo_df = pd.DataFrame(rows)
    halo_df.to_pickle(config.analysis_dir / "halo_analysis.pkl")
    return halo_df

## Section 20 — Portable Export Helper

In [ ]:
def _save_portable_table(df: pd.DataFrame, out_stem: Path) -> dict:
    out_stem.parent.mkdir(parents=True, exist_ok=True)
    saved: dict = {}
    if df is None:
        return saved
    csv_path = out_stem.with_suffix(".csv")
    df.to_csv(csv_path, index=False)
    saved["csv"] = str(csv_path)
    try:
        parquet_path = out_stem.with_suffix(".parquet")
        df.to_parquet(parquet_path, index=False)
        saved["parquet"] = str(parquet_path)
    except Exception as exc:
        pkl_path = out_stem.with_suffix(".pkl")
        df.to_pickle(pkl_path)
        saved["pickle"] = str(pkl_path)
        saved["parquet_error"] = str(exc)
    return saved


def export_portable_analysis_bundle(
    config: PipelineConfig,
    export_dir: Union[str, Path],
    seg_index_df:  Optional[pd.DataFrame] = None,
    objects_df:    Optional[pd.DataFrame] = None,
    grouped_z_df:  Optional[pd.DataFrame] = None,
    best_z_df:     Optional[pd.DataFrame] = None,
    filtered_df:   Optional[pd.DataFrame] = None,
    timed_df:      Optional[pd.DataFrame] = None,
    tracked_df:    Optional[pd.DataFrame] = None,
    halo_df:       Optional[pd.DataFrame] = None,
    copy_mask_npy_files: bool = False,
) -> Path:
    export_dir = Path(export_dir)
    dirs = {k: export_dir / k for k in ("tables", "masks", "config", "notes")}
    for p in [export_dir] + list(dirs.values()):
        p.mkdir(parents=True, exist_ok=True)

    manifest = {
        "created_utc":                  datetime.utcnow().isoformat(timespec="seconds") + "Z",
        "input_image_name":             config.input_image_name,
        "input_image_path_on_cluster":  str(config.input_image_path),
        "pixel_size_um":                float(config.pixel_size_um),
        "z_step_um":                    float(config.z_step_um),
        "nuclear_channel_index":        int(config.nuclear_channel_index),
        "membrane_channel_index":       int(config.membrane_channel_index),
        "prob_norm_coarse_threshold":   float(config.prob_norm_coarse_threshold),
        "prob_norm_final_threshold":    float(config.prob_norm_final_threshold),
        "copied_files": {}, "tables": {},
    }

    config_path = dirs["config"] / "pipeline_config.json"
    config_path.write_text(json.dumps(config.to_serializable_dict(), indent=2))
    manifest["copied_files"]["pipeline_config"] = str(config_path)

    tiff_exports = {
        "segmentation_class_hyperstack":        config.segmentation_class_hyperstack_path,
        "segmentation_probability_raw":         config.segmentation_probability_hyperstack_path,
        "segmentation_probability_normalised":  config.normalised_probability_hyperstack_path,
        "segmentation_label_hyperstack":        config.segmentation_label_hyperstack_path,
        "nucleus_instance_hyperstack":          config.nucleus_instance_hyperstack_path,
        "droplet_instance_hyperstack":          config.droplet_instance_hyperstack_path,
    }
    for key, src in tiff_exports.items():
        src = Path(src)
        if src.exists():
            dst = dirs["masks"] / src.name
            shutil.copy2(src, dst)
            manifest["copied_files"][key] = str(dst)

    for name, df in {
        "segmentation_index":           seg_index_df,
        "plane_objects":                objects_df,
        "grouped_z_objects":            grouped_z_df,
        "best_z_nuclei":                best_z_df,
        "best_z_nuclei_with_exclusion": filtered_df,
        "best_z_nuclei_timed":          timed_df,
        "tracked_nuclei":               tracked_df,
        "halo_analysis":                halo_df,
    }.items():
        if df is not None:
            manifest["tables"][name] = _save_portable_table(df, dirs["tables"] / name)

    if copy_mask_npy_files and seg_index_df is not None and "mask_path" in seg_index_df.columns:
        plane_mask_dir = dirs["masks"] / "plane_npy_masks"
        plane_mask_dir.mkdir(parents=True, exist_ok=True)
        portable_idx = seg_index_df.copy()
        new_paths = []
        for row in portable_idx.itertuples(index=False):
            src = Path(row.mask_path)
            if src.exists():
                dst = plane_mask_dir / src.name
                shutil.copy2(src, dst)
                new_paths.append(str(dst))
            else:
                new_paths.append(None)
        portable_idx["portable_mask_path"] = new_paths
        manifest["tables"]["segmentation_index_portable"] = _save_portable_table(
            portable_idx, dirs["tables"] / "segmentation_index_portable")

    readme = (
        "Portable laptop analysis bundle\n\n"
        "Contents\n--------\n"
        "masks/  — Hyperstack TIFFs: raw probability, per-object normalised probability,\n"
        "          class mask, instance labels.  Open in napari or Fiji for QC.\n"
        "tables/ — Analysis tables (CSV + parquet).\n"
        "config/ — Pipeline configuration snapshot.\n\n"
        "Recommended workflow\n--------------------\n"
        "1. Compare raw vs normalised probability TIFFs in napari to validate normalisation.\n"
        "2. Load halo_analysis.parquet for plotting and filtering.\n"
        "3. Use tracked_nuclei / best_z_nuclei for re-analysis without re-running the model.\n"
    )
    (dirs["notes"] / "README.txt").write_text(readme)
    (export_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
    return export_dir

## Section 21 — QC Plotting Helpers

In [ ]:
# ── Segmentation QC ───────────────────────────────────────────────────────────

def plot_nucleus_mask_overlay_qc(
    img_5d: np.ndarray,
    nucleus_mask_4d: np.ndarray,
    config: PipelineConfig,
    timepoints=None,
    z_mode: str = "max_mask",
    max_cols: int = 4,
) -> None:
    """Overlay raw nuclear channel with the binary nucleus mask."""
    T, Z, C, Y, X = img_5d.shape
    if timepoints is None:
        timepoints = np.linspace(0, T - 1, min(T, 6), dtype=int).tolist()
    n_cols = min(max_cols, len(timepoints))
    n_rows = math.ceil(len(timepoints) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
    axes = np.atleast_1d(axes).ravel()
    for ax, t_idx in zip(axes, timepoints):
        z_idx = (
            Z // 2 if z_mode == "middle"
            else int(np.argmax(nucleus_mask_4d[t_idx].reshape(Z, -1).sum(axis=1)))
        )
        raw = img_5d[t_idx, z_idx, config.nuclear_channel_index].astype(np.float32)
        if raw.max() > 0:
            raw /= raw.max()
        ax.imshow(raw, cmap="gray")
        ax.imshow(nucleus_mask_4d[t_idx, z_idx].astype(bool), alpha=0.35)
        ax.set_title(f"t={t_idx}, z={z_idx}")
        ax.axis("off")
    for ax in axes[len(timepoints):]:
        ax.axis("off")
    fig.suptitle("QC: Nuclear channel + nucleus mask overlay", fontsize=14)
    plt.tight_layout()
    plt.show()


def plot_nucleus_mask_qc_summary(
    nucleus_mask_4d: np.ndarray,
    timepoints=None,
    z_mode: str = "max_mask",
    max_cols: int = 4,
) -> None:
    """Mask snapshots with contours + total nuclear area over time."""
    from skimage.measure import find_contours
    T, Z, Y, X = nucleus_mask_4d.shape
    if timepoints is None:
        timepoints = np.linspace(0, T - 1, min(T, 6), dtype=int).tolist()
    n_cols = min(max_cols, len(timepoints))
    n_rows = math.ceil(len(timepoints) / n_cols)
    fig = plt.figure(figsize=(4 * n_cols, 4 * (n_rows + 1)))
    gs  = fig.add_gridspec(n_rows + 1, n_cols)
    panel_axes = [fig.add_subplot(gs[r, c]) for r in range(n_rows) for c in range(n_cols)]
    for ax, t_idx in zip(panel_axes, timepoints):
        z_idx = (
            Z // 2 if z_mode == "middle"
            else int(np.argmax(nucleus_mask_4d[t_idx].reshape(Z, -1).sum(axis=1)))
        )
        mask_plane = nucleus_mask_4d[t_idx, z_idx].astype(bool)
        labeled    = measure.label(mask_plane)
        ax.imshow(mask_plane, cmap="gray")
        for contour in find_contours(mask_plane.astype(float), level=0.5):
            ax.plot(contour[:, 1], contour[:, 0], linewidth=0.5)
        ax.set_title(f"t={t_idx}, z={z_idx}\nobjects={labeled.max()}")
        ax.axis("off")
    for ax in panel_axes[len(timepoints):]:
        ax.axis("off")
    area_ax = fig.add_subplot(gs[n_rows, :])
    area_ax.plot(np.arange(T), nucleus_mask_4d.reshape(T, -1).sum(axis=1), marker="o")
    area_ax.set_xlabel("Time index")
    area_ax.set_ylabel("Total nucleus mask area (px)")
    area_ax.set_title("QC: Total segmented nuclear area over time")
    plt.tight_layout()
    plt.show()


def plot_nucleus_count_over_time(nucleus_mask_4d: np.ndarray) -> None:
    """Count connected nucleus objects across all Z slices per time point."""
    T, Z = nucleus_mask_4d.shape[:2]
    counts = [
        sum(measure.label(nucleus_mask_4d[t, z].astype(bool)).max() for z in range(Z))
        for t in range(T)
    ]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(np.arange(T), counts, marker="o")
    ax.set_xlabel("Time index")
    ax.set_ylabel("Connected nucleus objects")
    ax.set_title("QC: Connected nucleus object count over time")
    plt.tight_layout()
    plt.show()


def plot_best_focus_nuclear_planes(
    img_5d: np.ndarray,
    config: PipelineConfig,
    figsize_per_panel: float = 4.0,
) -> pd.DataFrame:
    """Plot the single best-focus nuclear-channel Z plane for each time point."""
    n_t = img_5d.shape[0]
    n_cols = min(5, n_t)
    n_rows = math.ceil(n_t / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(figsize_per_panel * n_cols, figsize_per_panel * n_rows))
    axes = np.array(axes).reshape(-1)
    focus_rows = []
    for t in range(n_t):
        keep_z, scores, best_z = get_best_focus_z_indices(
            img_5d, t,
            nuc_channel_idx=config.effective_focus_channel_index,
            exclude_edge_z=config.focus_edge_z_exclusion,
            window_radius=0,
            metric=config.focus_metric,
            pixel_size_um=config.pixel_size_um,
            config=config,
        )
        ax = axes[t]
        if best_z is None:
            ax.set_title(f"t={t}: no valid focus plane")
            ax.axis("off")
            focus_rows.append({"t": t, "best_z": np.nan, "focus_score": np.nan,
                                "focus_metric": config.focus_metric})
            continue
        nuc2d      = img_5d[t, best_z, config.effective_focus_channel_index]
        best_score = float(scores[best_z])
        ax.imshow(nuc2d, cmap="gray")
        ax.set_title(f"t={t}, best z={best_z}\nfocus={best_score:.4g}")
        ax.axis("off")
        focus_rows.append({"t": t, "best_z": int(best_z), "focus_score": best_score,
                           "focus_metric": config.focus_metric})
    for ax in axes[n_t:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    return pd.DataFrame(focus_rows)


# ── Probability normalisation QC ──────────────────────────────────────────────

def plot_probability_normalisation_qc(
    raw_prob: np.ndarray,
    norm_prob: np.ndarray,
    class_idx: int,
    class_name: str = "nucleus",
    final_threshold: float = 0.5,
    title: str = "",
) -> None:
    """
    Side-by-side comparison of raw vs per-object normalised probability maps
    with histograms, so you can visually validate that normalisation is making
    within-object probabilities uniform before thresholding.

    Parameters
    ----------
    raw_prob        : (Y, X, C) raw U-Net probability map
    norm_prob       : (Y, X, C) per-object normalised probability map
    class_idx       : class channel to inspect
    class_name      : label for plot titles
    final_threshold : shown as a vertical line on histograms
    """
    raw_ch  = raw_prob [..  , class_idx]
    norm_ch = norm_prob[..  , class_idx]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))

    # Row 0: probability maps
    im0 = axes[0, 0].imshow(raw_ch,  cmap="inferno", vmin=0, vmax=1)
    axes[0, 0].set_title(f"Raw probability — {class_name}")
    axes[0, 0].axis("off")
    plt.colorbar(im0, ax=axes[0, 0])

    im1 = axes[0, 1].imshow(norm_ch, cmap="inferno", vmin=0, vmax=1)
    axes[0, 1].set_title(f"Normalised probability — {class_name}")
    axes[0, 1].axis("off")
    plt.colorbar(im1, ax=axes[0, 1])

    # Difference map
    diff = norm_ch - raw_ch
    lim  = max(abs(diff.min()), abs(diff.max()), 1e-6)
    imd  = axes[0, 2].imshow(diff, cmap="RdBu_r", vmin=-lim, vmax=lim)
    axes[0, 2].set_title("Difference (norm - raw)")
    axes[0, 2].axis("off")
    plt.colorbar(imd, ax=axes[0, 2])

    # Row 1: histograms
    for ax, ch, label in [
        (axes[1, 0], raw_ch,  "Raw"),
        (axes[1, 1], norm_ch, "Normalised"),
    ]:
        # Exclude true zeros (background / non-normalised pixels)
        nonzero = ch[ch > 0]
        ax.hist(nonzero, bins=100, color="steelblue", edgecolor="none")
        ax.axvline(final_threshold, color="red", linestyle="--",
                   label=f"threshold={final_threshold}")
        ax.set_xlabel("Probability (non-zero pixels)")
        ax.set_ylabel("Count")
        ax.set_title(f"{label} probability distribution")
        ax.legend(fontsize=8)

    # Scatter: raw vs normalised (sampled for speed)
    flat_raw  = raw_ch.ravel()
    flat_norm = norm_ch.ravel()
    nonzero_idx = np.where((flat_raw > 0) | (flat_norm > 0))[0]
    sample_idx  = np.random.choice(nonzero_idx, size=min(5000, len(nonzero_idx)), replace=False)
    axes[1, 2].scatter(flat_raw[sample_idx], flat_norm[sample_idx],
                       alpha=0.3, s=5, color="steelblue")
    axes[1, 2].plot([0, 1], [0, 1], "k--", linewidth=0.8, label="y=x")
    axes[1, 2].axhline(final_threshold, color="red", linestyle="--", linewidth=0.8)
    axes[1, 2].set_xlabel("Raw probability")
    axes[1, 2].set_ylabel("Normalised probability")
    axes[1, 2].set_title("Raw vs Normalised (sampled)")
    axes[1, 2].legend(fontsize=8)

    if title:
        fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()


def plot_per_object_probability_uniformity(
    raw_prob_2d: np.ndarray,
    norm_prob_2d: np.ndarray,
    labeled_mask: np.ndarray,
    max_objects: int = 30,
    title: str = "",
) -> None:
    """
    Boxplot of intra-object probability variance before and after normalisation
    for each detected object.  A successful normalisation should produce near-zero
    variance in the normalised map for all objects regardless of their raw signal.

    Parameters
    ----------
    raw_prob_2d  : (Y, X) raw probability for one class
    norm_prob_2d : (Y, X) normalised probability for the same class
    labeled_mask : (Y, X) integer label map (from measure.label)
    max_objects  : cap displayed objects so the plot remains readable
    """
    labels = [p.label for p in measure.regionprops(labeled_mask)]
    if not labels:
        print("No objects found in labeled_mask.")
        return
    labels = labels[:max_objects]

    raw_vars, norm_vars = [], []
    for lbl in labels:
        obj_mask = labeled_mask == lbl
        raw_vars.append(float(np.var(raw_prob_2d [obj_mask])))
        norm_vars.append(float(np.var(norm_prob_2d[obj_mask])))

    x = np.arange(len(labels))
    fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)

    for ax, variances, label in [
        (axes[0], raw_vars,  "Raw intra-object variance"),
        (axes[1], norm_vars, "Normalised intra-object variance"),
    ]:
        ax.bar(x, variances, color="steelblue", edgecolor="none")
        ax.axhline(np.mean(variances), color="red", linestyle="--",
                   label=f"mean={np.mean(variances):.4f}")
        ax.set_xlabel("Object index")
        ax.set_ylabel("Intra-object probability variance")
        ax.set_title(label)
        ax.legend(fontsize=8)

    if title:
        fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()


# ── Downstream measurement plots ──────────────────────────────────────────────

def plot_nc_vs_time(halo_df: pd.DataFrame) -> None:
    if halo_df.empty:
        print("No data to plot.")
        return
    fig, ax = plt.subplots(figsize=(7, 4))
    for track_id, df_t in halo_df.groupby("track_id"):
        df_t = df_t.sort_values("true_time_min")
        ax.plot(df_t["true_time_min"], df_t["nc_ratio_fraction"], marker="o", label=f"Track {track_id}")
    ax.set_xlabel("True acquisition time (min)")
    ax.set_ylabel("N / (N + C)")
    ax.set_title("N/C ratio fraction by track")
    plt.tight_layout()
    plt.show()


def plot_area_vs_time(halo_df: pd.DataFrame) -> None:
    if halo_df.empty:
        print("No data to plot.")
        return
    fig, ax = plt.subplots(figsize=(7, 4))
    for track_id, df_t in halo_df.groupby("track_id"):
        df_t = df_t.sort_values("true_time_min")
        ax.plot(df_t["true_time_min"], df_t["nucleus_area_um2"], marker="o", label=f"Track {track_id}")
    ax.set_xlabel("True acquisition time (min)")
    ax.set_ylabel("Nuclear area (µm²)")
    ax.set_title("Largest nuclear cross-sectional area by track")
    plt.tight_layout()
    plt.show()


def plot_largest_cross_sectional_area_vs_time(
    df: pd.DataFrame,
    config: PipelineConfig,
) -> None:
    """Scatter all individual nuclear areas + population mean vs true acquisition time."""
    if df.empty:
        print("No data to plot.")
        return
    plot_df = df.copy()
    if "nucleus_area_um2" not in plot_df.columns:
        if "area_px" not in plot_df.columns:
            raise ValueError("DataFrame must contain 'nucleus_area_um2' or 'area_px'.")
        plot_df["nucleus_area_um2"] = plot_df["area_px"] * config.pixel_size_um ** 2
    if "true_time_min" not in plot_df.columns:
        raise ValueError("DataFrame must contain 'true_time_min'.")
    plot_df = plot_df.sort_values("true_time_min")
    mean_df = (
        plot_df.groupby("true_time_min", as_index=False)["nucleus_area_um2"]
        .mean().rename(columns={"nucleus_area_um2": "mean_area"})
    )
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(plot_df["true_time_min"], plot_df["nucleus_area_um2"],
               alpha=0.35, s=20, label="Individual nuclei")
    ax.plot(mean_df["true_time_min"], mean_df["mean_area"],
            linewidth=2, label="Population mean")
    ax.set_xlabel("True acquisition time (min)")
    ax.set_ylabel("Nuclear cross-sectional area (µm²)")
    ax.set_title("Nuclear area vs time")
    ax.legend()
    plt.tight_layout()
    plt.show()

## Section 22 — Stage Execution Flags

Set a flag to `True` to recompute that stage; `False` loads the saved artefact from disk.
This lets you re-run only the stage being debugged without repeating expensive steps.

In [ ]:
RUN_SEGMENTATION      = True
RUN_OBJECT_EXTRACTION = True
RUN_GROUP_ACROSS_Z    = True
RUN_SELECT_BEST_Z     = True
RUN_FLAG_CLOSE_NUCLEI = True
RUN_ADD_TIMING        = True
RUN_TRACKING          = True
RUN_HALO_ANALYSIS     = True
RUN_EXPORT_PORTABLE   = True

# ── Debug plane indices ───────────────────────────────────────────────────────
DEBUG_T_IDX = 0
DEBUG_Z_IDX = 0

print("Stage flags set.")

## Section 23 — Initialise Runtime and Confirm Paths

In [ ]:
configure_tensorflow_for_gpu(cfg)

print(f"Model path  : {cfg.model_path}")
print(f"Input image : {cfg.input_image_path}")
print(f"Output dir  : {cfg.output_dir}")
print(f"Seg dir     : {cfg.seg_dir}")
print(f"Mask TIF dir: {cfg.mask_tif_dir}")

assert cfg.model_path.exists(),       f"Missing model: {cfg.model_path}"
assert cfg.input_image_path.exists(), f"Missing image: {cfg.input_image_path}"

save_config(cfg, cfg.output_dir / "runtime_pipeline_config.json")
print("Runtime configuration saved.")

## Section 24 — Load Image

In [ ]:
t0 = time.perf_counter()
img_5d = load_image_5d(cfg.input_image_path)
print(f"Shape: {img_5d.shape}  |  dtype: {img_5d.dtype}  |  load time: {time.perf_counter() - t0:.2f}s")

T, Z, C, Y, X = img_5d.shape
print(f"T={T}  Z={Z}  C={C}  Y={Y}  X={X}")

## Section 25 — Focus-Score QC

In [ ]:
# Quick sanity-check: score a single time point across all Z planes.
keep_z, scores, best_z = get_best_focus_z_indices(
    img_5d=img_5d, t=DEBUG_T_IDX,
    nuc_channel_idx=cfg.effective_focus_channel_index,
    exclude_edge_z=cfg.focus_edge_z_exclusion,
    window_radius=cfg.focus_window_radius,
    metric=cfg.focus_metric,
    pixel_size_um=cfg.pixel_size_um,
    config=cfg,
)
print(f"Best Z: {best_z}  |  Keep Z: {keep_z}")
print(f"Scores: {np.round(scores, 4)}")

plot_focus_scores_for_timepoint(img_5d, DEBUG_T_IDX, cfg)

# Uncomment to inspect all time points:
# focus_qc_df = plot_best_focus_nuclear_planes(img_5d=img_5d, config=cfg)
# display(focus_qc_df)

## Section 26 — Single-Plane Segmentation Debug

In [ ]:
# Load the model once; Section 27 (full segmentation) reuses it.
model = None

if RUN_SEGMENTATION:
    model = load_unet_model(cfg.model_path)
    print("Model loaded.")

plane_yxc = get_full_plane_yxc(img_5d, DEBUG_T_IDX, DEBUG_Z_IDX)
print(f"Debug plane shape: {plane_yxc.shape}")

if model is not None:
    cfg.batch_size = 1  # memory-safe for debug
    debug_raw_prob, debug_class_map, debug_nucleus_mask, debug_instance_labels, debug_norm_prob = \
        segment_single_plane_with_overlap(
            plane_yxc, model, cfg,
            patch_size=cfg.patch_size, stride=cfg.patch_stride,
        )
    print("Single-plane debug segmentation complete.")
    print(f"  Raw probability map shape  : {debug_raw_prob.shape}")
    print(f"  Normalised probability shape: {debug_norm_prob.shape}")
    print(f"  Unique class labels         : {np.unique(debug_class_map)}")
    print(f"  Nucleus pixels              : {int(debug_nucleus_mask.sum())}")
    gc.collect()

## Section 27 — Probability Normalisation QC (Debug Plane)

In [ ]:
# Visualise the effect of per-object normalisation on the debug plane
# before committing to a full segmentation run.

if "debug_raw_prob" in dir() and debug_raw_prob is not None:
    # Nucleus channel
    plot_probability_normalisation_qc(
        raw_prob=debug_raw_prob,
        norm_prob=debug_norm_prob,
        class_idx=cfg.nucleus_class_index,
        class_name="nucleus",
        final_threshold=cfg.prob_norm_final_threshold,
        title=f"Normalisation QC — nucleus — t={DEBUG_T_IDX}, z={DEBUG_Z_IDX}",
    )

    # Droplet channel
    plot_probability_normalisation_qc(
        raw_prob=debug_raw_prob,
        norm_prob=debug_norm_prob,
        class_idx=cfg.droplet_class_index,
        class_name="droplet",
        final_threshold=cfg.prob_norm_final_threshold,
        title=f"Normalisation QC — droplet — t={DEBUG_T_IDX}, z={DEBUG_Z_IDX}",
    )

    # Per-object variance comparison (nucleus channel)
    labeled_debug = measure.label(
        debug_norm_prob[..., cfg.nucleus_class_index] > cfg.prob_norm_coarse_threshold,
        connectivity=2,
    )
    plot_per_object_probability_uniformity(
        raw_prob_2d=debug_raw_prob[..., cfg.nucleus_class_index],
        norm_prob_2d=debug_norm_prob[..., cfg.nucleus_class_index],
        labeled_mask=labeled_debug,
        title=f"Per-object variance — nucleus — t={DEBUG_T_IDX}, z={DEBUG_Z_IDX}",
    )
else:
    print("Run Section 26 first (single-plane debug).")

## Section 28 — Run or Load Full Segmentation

In [ ]:
if RUN_SEGMENTATION:
    if model is None:
        model = load_unet_model(cfg.model_path)
    t0 = time.perf_counter()
    seg_index_df = run_segmentation_for_all_planes(
        img_5d=img_5d, model=model, config=cfg,
        save_binary_masks=True, save_probability_tiff=True,
    )
    print(f"Segmentation finished in {time.perf_counter() - t0:.2f}s")
else:
    seg_index_df = pd.read_pickle(cfg.segmentation_index_path)
    print("Loaded saved segmentation index.")

print(seg_index_df.shape)
display(seg_index_df.head())

## Section 29 — Run or Load Object Extraction

In [ ]:
objects_path = cfg.obj_dir / "plane_objects.pkl"

if RUN_OBJECT_EXTRACTION:
    t0 = time.perf_counter()
    objects_df = extract_objects_from_saved_masks(seg_index_df, cfg)
    print(f"Object extraction finished in {time.perf_counter() - t0:.2f}s")
else:
    objects_df = pd.read_pickle(objects_path)
    print("Loaded saved object table.")

print(objects_df.shape)
display(objects_df.head())

## Section 30 — Run or Load Z Grouping

In [ ]:
grouped_z_path = cfg.obj_dir / "grouped_z_objects.pkl"

if RUN_GROUP_ACROSS_Z:
    t0 = time.perf_counter()
    grouped_z_df = group_nuclei_across_z(objects_df, cfg)
    print(f"Z grouping finished in {time.perf_counter() - t0:.2f}s")
else:
    grouped_z_df = pd.read_pickle(grouped_z_path)
    print("Loaded grouped-Z table.")

print(grouped_z_df.shape)
display(grouped_z_df.head())

## Section 31 — Run or Load Best-Z Selection

In [ ]:
best_z_path = cfg.obj_dir / "best_z_nuclei.pkl"

if RUN_SELECT_BEST_Z:
    t0 = time.perf_counter()
    best_z_df = select_best_z_per_nucleus(grouped_z_df, cfg)
    print(f"Best-Z selection finished in {time.perf_counter() - t0:.2f}s")
else:
    best_z_df = pd.read_pickle(best_z_path)
    print("Loaded best-Z table.")

print(best_z_df.shape)
display(best_z_df.head())

## Section 32 — Optional: Area-Based Cleanup

In [ ]:
def filter_nuclei_by_area(
    df: pd.DataFrame,
    config: PipelineConfig,
    min_area_um2: float = 100.0,
    max_area_um2: float = 300.0,
) -> pd.DataFrame:
    """Remove objects that are too small or too large to be real nuclei."""
    df = df.copy()
    if "nucleus_area_um2" not in df.columns:
        df["nucleus_area_um2"] = df["area_px"] * config.pixel_size_um ** 2
    filtered = df[
        (df["nucleus_area_um2"] >= min_area_um2) &
        (df["nucleus_area_um2"] <= max_area_um2)
    ]
    print(f"Area filter: {len(df)} → {len(filtered)} nuclei")
    return filtered


# Uncomment and tune to activate:
# best_z_df = filter_nuclei_by_area(best_z_df, cfg, min_area_um2=100, max_area_um2=300)
# display(best_z_df.head())

## Section 33 — Run or Load Close-Nuclei Exclusion

In [ ]:
filtered_path = cfg.obj_dir / "best_z_nuclei_with_exclusion.pkl"

if RUN_FLAG_CLOSE_NUCLEI:
    t0 = time.perf_counter()
    filtered_df = flag_close_nuclei(best_z_df, cfg)
    print(f"Close-nuclei flagging finished in {time.perf_counter() - t0:.2f}s")
else:
    filtered_df = pd.read_pickle(filtered_path)
    print("Loaded exclusion table.")

n_valid = int(filtered_df["valid_single_nucleus"].sum())
print(f"{n_valid} / {len(filtered_df)} nuclei flagged as valid single-nucleus droplets")
display(filtered_df.head())

## Section 34 — Run or Load Timing Metadata

In [ ]:
timed_path = cfg.track_dir / "best_z_nuclei_timed.pkl"

if RUN_ADD_TIMING:
    t0 = time.perf_counter()
    timed_df = add_tile_timing_metadata(filtered_df, img_5d, cfg)
    print(f"Timing metadata finished in {time.perf_counter() - t0:.2f}s")
else:
    timed_df = pd.read_pickle(timed_path)
    print("Loaded timed nuclei table.")

print(timed_df.shape)
display(timed_df.head())

## Section 35 — Run or Load Tracking

In [ ]:
tracked_path = cfg.track_dir / "tracked_nuclei.pkl"

if RUN_TRACKING:
    t0 = time.perf_counter()
    tracked_df = assign_track_ids_hybrid_dbscan(timed_df, cfg, valid_only=False)
    print(f"Hybrid DBSCAN tracking finished in {time.perf_counter() - t0:.2f}s")
else:
    tracked_df = pd.read_pickle(tracked_path)
    print("Loaded tracked nuclei table.")

print(tracked_df.shape)
display(tracked_df.head())
display(summarize_tracking_debug(tracked_df))

## Section 36 — Run or Load Halo Analysis

In [ ]:
halo_path = cfg.analysis_dir / "halo_analysis.pkl"

if RUN_HALO_ANALYSIS:
    t0 = time.perf_counter()
    halo_df = run_halo_analysis_for_tracked_nuclei(tracked_df, img_5d, cfg)
    print(f"Halo analysis finished in {time.perf_counter() - t0:.2f}s")
else:
    halo_df = pd.read_pickle(halo_path)
    print("Loaded halo analysis table.")

print(halo_df.shape)
display(halo_df.head())

## Section 37 — Load Saved Mask Hyperstacks for QC

In [ ]:
class_mask_5d       = tiff.imread(cfg.segmentation_class_hyperstack_path)
label_mask_4d       = tiff.imread(cfg.segmentation_label_hyperstack_path)
nucleus_instance_4d = tiff.imread(cfg.nucleus_instance_hyperstack_path)

print(f"class_mask_5d shape       : {class_mask_5d.shape}")
print(f"label_mask_4d shape       : {label_mask_4d.shape}")
print(f"nucleus_instance_4d shape : {nucleus_instance_4d.shape}")

nucleus_mask_4d = class_mask_5d[:, :, cfg.nucleus_class_index] > 0
print(f"nucleus_mask_4d shape     : {nucleus_mask_4d.shape}")

## Section 38 — Segmentation QC Plots

In [ ]:
plot_nucleus_mask_overlay_qc(img_5d, nucleus_mask_4d, cfg, z_mode="max_mask")
plot_nucleus_mask_qc_summary(nucleus_mask_4d, z_mode="max_mask")
plot_nucleus_count_over_time(nucleus_mask_4d)

## Section 39 — Downstream Measurement Plots

In [ ]:
plot_nc_vs_time(halo_df)
plot_area_vs_time(halo_df)
plot_largest_cross_sectional_area_vs_time(timed_df, cfg)

## Section 40 — Pipeline Summary

In [ ]:
summary_df = pd.DataFrame([{
    "seg_index_rows":  len(seg_index_df),
    "objects_rows":    len(objects_df),
    "grouped_z_rows":  len(grouped_z_df),
    "best_z_rows":     len(best_z_df),
    "filtered_rows":   len(filtered_df),
    "timed_rows":      len(timed_df),
    "tracked_rows":    len(tracked_df),
    "halo_rows":       len(halo_df),
    "n_tracks":        int(tracked_df["track_id"].nunique()) if not tracked_df.empty else 0,
}])
display(summary_df)

## Section 41 — Export Portable Analysis Bundle

In [ ]:
if RUN_EXPORT_PORTABLE:
    portable_dir = cfg.output_dir / "portable_laptop_bundle"
    exported_path = export_portable_analysis_bundle(
        config=cfg,
        export_dir=portable_dir,
        seg_index_df=seg_index_df,
        objects_df=objects_df,
        grouped_z_df=grouped_z_df,
        best_z_df=best_z_df,
        filtered_df=filtered_df,
        timed_df=timed_df,
        tracked_df=tracked_df,
        halo_df=halo_df,
        copy_mask_npy_files=False,
    )
    print(f"Portable bundle written to: {exported_path}")

## Section 42 — (Optional) Inspect Probability TIFFs

In [ ]:
for label, path in [
    ("Raw probability",        cfg.segmentation_probability_hyperstack_path),
    ("Normalised probability", cfg.normalised_probability_hyperstack_path),
]:
    if path.exists():
        arr = tiff.imread(path)
        print(f"{label}:")
        print(f"  Shape : {arr.shape}")
        print(f"  dtype : {arr.dtype}")
        print(f"  Range : {float(np.nanmin(arr)):.4f} – {float(np.nanmax(arr)):.4f}")
    else:
        print(f"{label}: not found at {path}")